In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Yang Hui\Desktop\projects\landy_ai\Open Transaction Data.csv', 
                 encoding='utf-16', 
                 sep='\t') # Try '\t' or ',' or ';' based on file structure



In [41]:
import pandas as pd
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
from tqdm import tqdm

# ── Cache (persists across runs) ───────────────────────────────────────────────
set_llm_cache(SQLiteCache(database_path="llm_cache.db"))

# ── Schema ─────────────────────────────────────────────────────────────────────
class Coordinates(TypedDict):
    latitude: float
    longitude: float
    
from dotenv import load_dotenv
load_dotenv()
# ── LLM setup ──────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=os.environ.get("AI_GATEWAY_API_KEY"),
    base_url="https://ai-gateway.vercel.sh/v1"
    ).with_structured_output(Coordinates)

# ── Address builder ────────────────────────────────────────────────────────────
def build_address(row: pd.Series) -> str:
    parts = [
        row.get("Road Name", ""),
        row.get("Scheme Name/Area", ""),
        row.get("Mukim", ""),
        row.get("District", ""),
        "Malaysia",
    ]
    return " ".join(p.strip() for p in parts if str(p).strip() not in ("", "nan"))

# ── Geocode one address ────────────────────────────────────────────────────────
def geocode(address: str) -> Coordinates | None:
    try:
        return llm.invoke(f"coordinates for {address}")
    except Exception as e:
        print(f"  [error] {address[:60]}... → {e}")
        return None



In [42]:
COORDS_CACHE = "coords_cache.csv"   # incremental save — survives crashes
OUTPUT_FILE  = "property_geocoded.csv"
BATCH_SIZE   = 50
# ── Main ───────────────────────────────────────────────────────────────────────
df = pd.read_csv(r'C:\Users\Yang Hui\Desktop\projects\landy_ai\Open Transaction Data.csv', 
                 encoding='utf-16', 
                 sep='\t') # Try '\t' or ',' or ';' based on file structure

df["address"] = df.apply(build_address, axis=1)
 
unique = df[["address"]].drop_duplicates().reset_index(drop=True)
 
# Resume from previous run if coords_cache.csv exists
import os
if os.path.exists(COORDS_CACHE):
    done_df  = pd.read_csv(COORDS_CACHE)
    done_set = set(done_df["address"])
    print(f"[resume] {len(done_set):,} already cached, skipping...")
else:
    done_df  = pd.DataFrame(columns=["address", "latitude", "longitude"])
    done_set = set()
 
todo = unique[~unique["address"].isin(done_set)].reset_index(drop=True)
print(f"[todo]   {len(todo):,} addresses to geocode")
 
# Process in batches, saving after each batch
batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
 
for batch_idx, batch in enumerate(tqdm(batches, desc="Batches")):
    results = []
    for addr in batch["address"]:
        coord = geocode(addr)
        results.append({
            "address":   addr,
            "latitude":  coord["latitude"]  if coord else None,
            "longitude": coord["longitude"] if coord else None,
        })
 
    batch_df = pd.DataFrame(results)
    done_df  = pd.concat([done_df, batch_df], ignore_index=True)
 
    # Save after every batch — safe to interrupt anytime
    done_df.to_csv(COORDS_CACHE, index=False)
 
# Merge final coords back to original dataframe
df = df.merge(done_df, on="address", how="left")
df.to_csv(OUTPUT_FILE, index=False)
print(f"Done — {df['latitude'].notna().sum():,}/{len(df):,} geocoded")

[todo]   8,514 addresses to geocode


Batches: 100%|██████████| 171/171 [2:54:08<00:00, 61.10s/it]  

Done — 23,808/23,808 geocoded


In [43]:
df

,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,Unit,Unit Level,Transaction Price,Unnamed: 13,address,latitude,longitude
0,1 - 1 1/2 Storey Shop,Gombak,Bandar Kepong,TAMAN INDAH PERDANA,JALAN PERDANA 7,September 2024,Leasehold,121.00,sq.m,108,sq.m,,"RM1,000,000.00",NaN,JALAN PERDANA 7 TAMAN INDAH PERDANA Bandar Kep...,3.2337,101.6771
1,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,June 2023,Freehold,185.80,sq.m,111,sq.m,,"RM340,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143
2,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,June 2023,Freehold,185.80,sq.m,111,sq.m,,"RM350,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143
3,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,November 2023,Freehold,185.80,sq.m,111,sq.m,,"RM320,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143
4,1 - 1 1/2 Storey Shop,Gombak,Bandar Rawang,KG DATO LEE KIM SAI,JALAN BATU 18/3,June 2023,Leasehold,222.00,sq.m,222,sq.m,,"RM730,000.00",NaN,JALAN BATU 18/3 KG DATO LEE KIM SAI Bandar Raw...,3.321,101.574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23803,Terraced Factory/Warehouse,Sepang,Dengkil,TMN PERINDUSTRIAN MERANTI UTAMA,JLN PULAU MERANTI,June 2022,Leasehold,435.50,sq.m,845,sq.m,,"RM3,928,000.00",NaN,JLN PULAU MERANTI TMN PERINDUSTRIAN MERANTI UT...,2.896,101.667
23804,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,JALAN SIP 1,February 2022,Freehold,449.00,sq.m,292,sq.m,,"RM1,248,000.00",NaN,JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...,2.8103,101.7024
23805,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,JALAN SIP 1,March 2022,Freehold,334.00,sq.m,292,sq.m,,"RM840,000.00",NaN,JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...,2.8103,101.7024
23806,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,SEPANG INDUSTRI PARK,February 2022,Freehold,334.00,sq.m,292,sq.m,,"RM955,000.00",NaN,SEPANG INDUSTRI PARK SEPANG INDUSTRIAL PARK Pe...,2.7469,101.6915


In [60]:
df['Transaction Price  '][0].split("RM")

['', '1,000,000.00']

In [80]:
df['price'] = [int(x.split("RM")[-1].replace(",","").split(".")[0]) for x in df['Transaction Price  ']]
df['price']

0        1000000
1         340000
2         350000
3         320000
4         730000
          ...   
23803    3928000
23804    1248000
23805     840000
23806     955000
23807     780000
Name: price, Length: 23808, dtype: int64

In [89]:
df.keys()

Index(['Property Type', 'District', 'Mukim', 'Scheme Name/Area', 'Road Name',
       'Month, Year of Transaction Date', 'Tenure', 'Land/Parcel Area', 'Unit',
       'Main Floor Area', 'Unit        ', 'Unit Level', 'Transaction Price  ',
       'Unnamed: 13', 'address', 'latitude', 'longitude', 'price'],
      dtype='str')

In [100]:
df['Land/Parcel Area(sqft)'] = [int(float(x.replace(",", ""))*10.7639) for x in df['Land/Parcel Area']]
df['price/sqft'] = df['price']/df['Land/Parcel Area(sqft)']

In [109]:
df['year'] = [x.split(" ")[-1] for x in df['Month, Year of Transaction Date']]

In [113]:
df[['Property Type', 'price/sqft','District', 'Mukim', 'Scheme Name/Area', 'year']].groupby(['Property Type', 'District',  'year']).describe()

price/sqft                          \
                                              count        mean         std   
Property Type              District year                                      
1 - 1 1/2 Storey Shop      Gombak   2021        6.0  329.683736  112.603869   
                                    2022        5.0  379.243077  154.083812   
                                    2023        8.0  277.839118  128.606873   
                                    2024        4.0  640.180697  363.256007   
                                    2025        2.0  529.516024  122.106492   
...                                             ...         ...         ...   
Terraced Factory/Warehouse Sepang   2021        4.0  494.646116   57.417508   
                                    2022       29.0  534.776811  210.096720   
                                    2023       16.0  560.966250  246.088387   
                                    2024       12.0  512.752607  191.570907   
                                    2025        1.0  802.936453         NaN   

                                                                              \
                                                 min         25%         50%   
Property Type              District year                                       
1 - 1 1/2 Storey Shop      Gombak   2021  227.451667  265.903291  291.288801   
                                    2022  200.125078  307.017544  384.911470   
                                    2023  160.080040  173.836918  252.764617   
                                    2024  225.806452  427.456179  631.360955   
                                    2025  443.173695  486.344860  529.516024   
...                                              ...         ...         ...   
Terraced Factory/Warehouse Sepang   2021  452.209661  453.034283  475.974615   
                                    2022  233.657858  392.112421  532.967033   
                                    2023  231.449966  439.562500  548.374017   
                                    2024  199.800200  414.351061  524.320036   
                                    2025  802.936453  802.936453  802.936453   

                                                                   
                                                 75%          max  
Property Type              District year                           
1 - 1 1/2 Storey Shop      Gombak   2021  352.189868   538.141470  
                                    2022  384.911470   619.249823  
                                    2023  320.537513   540.915395  
                                    2024  844.085473  1072.194425  
                                    2025  572.687188   615.858353  
...                                              ...          ...  
Terraced Factory/Warehouse Sepang   2021  517.586448   574.425574  
                                    2022  639.360639   838.062727  
                                    2023  648.101898  1200.674536  
                                    2024  647.746406   779.220779  
                                    2025  802.936453   802.936453  

[457 rows x 8 columns]

In [123]:
len(np.unique(df['Mukim']))

160

In [122]:
df

,Property Type,District,Mukim,Scheme Name/Area,Road Name,"Month, Year of Transaction Date",Tenure,Land/Parcel Area,Unit,Main Floor Area,...,Unit Level,Transaction Price,Unnamed: 13,address,latitude,longitude,price,Land/Parcel Area(sqft),price/sqft,year
0,1 - 1 1/2 Storey Shop,Gombak,Bandar Kepong,TAMAN INDAH PERDANA,JALAN PERDANA 7,September 2024,Leasehold,121.00,sq.m,108,...,,"RM1,000,000.00",NaN,JALAN PERDANA 7 TAMAN INDAH PERDANA Bandar Kep...,3.2337,101.6771,1000000,1302,768.049155,2024
1,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,June 2023,Freehold,185.80,sq.m,111,...,,"RM340,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143,340000,1999,170.085043,2023
2,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,June 2023,Freehold,185.80,sq.m,111,...,,"RM350,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143,350000,1999,175.087544,2023
3,1 - 1 1/2 Storey Shop,Gombak,Bandar Kuang,BANDAR KUANG,JALAN PEKAN KUANG,November 2023,Freehold,185.80,sq.m,111,...,,"RM320,000.00",NaN,JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...,3.2595,101.6143,320000,1999,160.080040,2023
4,1 - 1 1/2 Storey Shop,Gombak,Bandar Rawang,KG DATO LEE KIM SAI,JALAN BATU 18/3,June 2023,Leasehold,222.00,sq.m,222,...,,"RM730,000.00",NaN,JALAN BATU 18/3 KG DATO LEE KIM SAI Bandar Raw...,3.321,101.574,730000,2389,305.567183,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23803,Terraced Factory/Warehouse,Sepang,Dengkil,TMN PERINDUSTRIAN MERANTI UTAMA,JLN PULAU MERANTI,June 2022,Leasehold,435.50,sq.m,845,...,,"RM3,928,000.00",NaN,JLN PULAU MERANTI TMN PERINDUSTRIAN MERANTI UT...,2.896,101.667,3928000,4687,838.062727,2022
23804,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,JALAN SIP 1,February 2022,Freehold,449.00,sq.m,292,...,,"RM1,248,000.00",NaN,JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...,2.8103,101.7024,1248000,4832,258.278146,2022
23805,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,JALAN SIP 1,March 2022,Freehold,334.00,sq.m,292,...,,"RM840,000.00",NaN,JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...,2.8103,101.7024,840000,3595,233.657858,2022
23806,Terraced Factory/Warehouse,Sepang,Pekan Bt Satu,SEPANG INDUSTRIAL PARK,SEPANG INDUSTRI PARK,February 2022,Freehold,334.00,sq.m,292,...,,"RM955,000.00",NaN,SEPANG INDUSTRI PARK SEPANG INDUSTRIAL PARK Pe...,2.7469,101.6915,955000,3595,265.646732,2022


In [3]:
from typing import TypedDict

class Coordinates(TypedDict):
    latitude: float
    longitude: float

In [ ]:
from utility.llm_init import load_llm

load_llm(model = 'openai/gpt-5.4-mini').with_structured_output(Coordinates).invoke('coordinates for JALAN PPS 1 SALAK SQUARE Pekan Salak Sepang')

{'latitude': 2.7698, 'longitude': 101.6867}

In [13]:
inputs = [
    "What is the capital of France?",
    "What is the capital of Japan?",
    "What is the capital of Nigeria?"
]

In [14]:
await load_llm().abatch(inputs)

[AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 13, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 6.825e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': None, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'cache_creation_input_tokens': 0, 'market_cost': 6.825e-05}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini', 'system_fingerprint': 'fp_akfouffaua', 'id': 'gen_01KMDME76APJNJYDC9AWEGBE40', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1b47-1ac4-7d50-99f6-a2b04e711b7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'i

In [ ]:
async def geocode_pipeline(addresses: List[str], batch_size: int = 50):
    results = []

    for chunk in chunk_list(addresses, batch_size):
        batch_result = await process_chunk(chunk)
        results.extend(batch_result)

    return results

async def process_chunk(addresses: List[str]):
    cached_results = []
    uncached_addresses = []

    # ---- Cache split ----
    for addr in addresses:
        if addr in cache:
            cached_results.append({
                "address": addr,
                "result": cache[addr],
                "source": "cache"
            })
        else:
            uncached_addresses.append(addr)

    # ---- If everything cached ----
    if not uncached_addresses:
        return cached_results

    # ---- Try abatch first ----
    try:
        prompts = [
            f"coordinates for {addr}"
            for addr in uncached_addresses
        ]

        responses = await llm.abatch(prompts)

        new_results = []
        for addr, res in zip(uncached_addresses, responses):
            if res:
                cache[addr] = res

            new_results.append({
                "address": addr,
                "result": res,
                "source": "abatch"
            })

    except Exception:
        # ---- Fallback to ainvoke ----
        new_results = await fallback_ainvoke(uncached_addresses)

    return cached_results + new_results

0        JALAN PERDANA 7 TAMAN INDAH PERDANA Bandar Kep...
1        JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...
2        JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...
3        JALAN PEKAN KUANG BANDAR KUANG Bandar Kuang Go...
4        JALAN BATU 18/3 KG DATO LEE KIM SAI Bandar Raw...
                               ...                        
23803    JLN PULAU MERANTI TMN PERINDUSTRIAN MERANTI UT...
23804    JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...
23805    JALAN SIP 1 SEPANG INDUSTRIAL PARK Pekan Bt Sa...
23806    SEPANG INDUSTRI PARK SEPANG INDUSTRIAL PARK Pe...
23807          JALAN PPS 1 SALAK SQUARE Pekan Salak Sepang
Length: 23808, dtype: str

In [19]:
def normalize(addr: str):
    return addr.strip().lower()